# Testing whether there is a systematic difference in CE for null distribution data

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

True


### Load data

In [3]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

(25424826, 13)

In [4]:
df.isna().sum()

Hxy                0
nx                 0
ny                 0
turn_delta         0
self               0
x_speaker          0
dyad               0
corpus             0
conversation_id    0
x_turn_id          0
null               0
groups             0
sample_delta       0
dtype: int64

In [5]:
df['conversation_id'].nunique()

2662

In [6]:
(df['turn_delta'] < 0).sum()

np.int64(61076)

In [7]:
df['corpus'].loc[(df['turn_delta'] < 0)].unique()

array(['se0-ag1', 'se0-ag2', 'se0-ag3', 'se0-ag4', 'se1-ag1', 'se1-ag3',
       'se2-ag1', 'se2-ag3', 'se2-ag4', 'se3-ag1', 'se3-ag2', 'se3-ag3'],
      dtype=object)

In [8]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,1
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,2
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,3
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,4
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,5


### Other data preparation processes

In [11]:
df['groups'].str.contains('null-').sum()

np.int64(2191693)

In [12]:
df['groups'] = [it.replace('null-', '') for it in tqdm(df['groups'])]

  0%|          | 0/25424826 [00:00<?, ?it/s]

In [13]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,1
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,2
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,3
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,4
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,5


### Convert values to numeric tags

In [14]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker',
    # 'dyad',
    'groups'
    #'x_speaker_turn'
]

In [15]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}

    if isinstance(col,list):
        for c  in col:
            df[c+'_'] = [conversion[v] for v in tqdm(df[c])]

    else:
        df[col+'_'] = [conversion[v] for v in tqdm(df[col])]


  0%|          | 0/25424826 [00:00<?, ?it/s]

## LME Regression: Predicting CE

In [16]:
import statsmodels.formula.api as smf

In [17]:
df_ = df

In [18]:
df_['null'].value_counts()

null
False    23233133
True      2191693
Name: count, dtype: int64

### Resid model

In [19]:
df_['g'] = 0

In [20]:
##########################################
## A set of resids to show ∇H / t
##########################################
# model = "Hxy ~ nx + ny"
model = "Hxy ~ nx + ny + null + (1|groups_)"
##########################################

In [21]:
start = dt.now()
# md = smf.mixedlm(model, data=df_, groups=df_['x_speaker_turn'])
md = smf.mixedlm(model, data=df_, groups=df_['g'])
mdf = md.fit()
print('completed in:', dt.now()-start)

completed in: 0:01:12.198399


In [23]:
# mdf.summary()
# mdf.bse

In [22]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

,coefs,stat,p,"CI[.025, .975]"
Intercept,-6.25e-01,-4.35e-02,9.65e-01,"[-2.88e+01, 2.75e+01]"
null[T.True],1.13e-01,4.07e+02,0.e+00,"[1.13e-01, 1.14e-01]"
nx,3.33e-01,5.20e+04,0.e+00,"[3.33e-01, 3.33e-01]"
ny,-1.28e-02,-1.99e+03,0.e+00,"[-1.28e-02, -1.27e-02]"
1 | groups_,1.5e-09,3.07e+00,2.13e-03,"[5.42e-10, 2.45e-09]"
Group Var,1.e+00,1.09e-03,9.99e-01,"[-1.80e+03, 1.80e+03]"


In [23]:
report_name = 'how-much-does-null-differ.csv'
report_name = os.path.join(DATA_LOC,report_name)

In [24]:
reporting.to_csv(
    report_name,
    encoding='utf-8'
)

In [25]:
reporting['Var'] = reporting.index.values

with open(report_name.replace('.csv', '.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()